## Imports

In [1]:
import pandas as pd
import re
from datetime import datetime, timedelta

## Initial Formatting

In [7]:
# Load raw data
df = pd.read_csv("data/mcp_desc_10_test.csv")

# --- 1. Remove emojis / non-ascii ---
def remove_non_ascii(text):
    if isinstance(text, str):
        return re.sub(r'[^\x00-\x7F]+', '', text)
    return text

for col in ["title", "description", "use_cases"]:
    df[col] = df[col].apply(remove_non_ascii).str.strip()

# --- 2. Drop junk / empty text ---
# heuristics for "useless" descriptions
def is_useless_desc(text):
    if not isinstance(text, str):
        return True
    txt = text.lower().strip()
    if len(txt.split()) < 5:
        return True
    bad_patterns = ["mirror of", "created from mcp server demo", "mcp-server", "demo", "test"]
    return any(p in txt for p in bad_patterns)

df["desc_is_useless"] = df["description"].apply(is_useless_desc)
df["has_use_cases"] = df["use_cases"].notna() & (df["use_cases"].str.strip() != "")

# keep if we have use cases OR meaningful description
df = df[(~df["desc_is_useless"]) | (df["has_use_cases"])].reset_index(drop=True)

# --- 3. Clean whitespace, collapse doubles ---
df = df.replace({r'\s+': ' '}, regex=True)

# --- 4. Standardize uploaded column ---

def parse_relative_date(text):
    if not isinstance(text, str):
        return None
    m = re.search(r"(\d+)\s*(month|week|day)s?\s*ago", text.lower())
    if not m:
        return None
    num, unit = int(m.group(1)), m.group(2)
    now = datetime.now()

    if unit == "month":
        # approximate months as 30 days each
        return now - timedelta(days=num * 30)
    elif unit == "week":
        return now - timedelta(weeks=num)
    elif unit == "day":
        return now - timedelta(days=num)
    else:
        return None

df["uploaded_clean"] = df["uploaded"].apply(parse_relative_date)
df["uploaded_days_ago"] = (datetime.now() - df["uploaded_clean"]).dt.days

# --- 5. Combine description + use cases for LLM input ---
def combine_text(desc, use):
    parts = []
    if isinstance(desc, str) and desc.strip():
        parts.append(f"Description: {desc.strip()}")
    if isinstance(use, str) and use.strip():
        parts.append(f"Use cases: {use.strip()}")
    return " ".join(parts)

df["text_for_llm"] = df.apply(lambda x: combine_text(x["description"], x["use_cases"]), axis=1)

# --- 6. Drop rows with no usable text at all ---
df = df[df["text_for_llm"].str.strip() != ""]

# --- 7. Optional: flag short entries ---
df["len_text"] = df["text_for_llm"].str.len()
df = df[df["len_text"] > 40]  # drop really short junk

# drop columns we don't need anymore
cols_to_drop = [
    "uploaded",          # raw text version
    "use_cases",         # now merged into text_for_llm
    "description",       # same, merged
    "desc_is_useless",   # helper boolean
    "has_use_cases"      # helper boolean
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# --- 8. Save cleaned dataset ---
df.to_csv("data/mcp_desc_10_test_cleaned.csv", index=False)
print(f"✅ Cleaned dataset saved ({len(df)} rows)")


✅ Cleaned dataset saved (410 rows)
